# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print("Dataset name:", metadata.get("name", ""))
print("Description:", metadata.get("description", ""))

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use `dataset.metadata.to_json()` to inspect the record sets and their fields. All references are made via their `@id`s.

In [ ]:
# Explore all record sets, fields, and their @id in the metadata
metadata_json = dataset.metadata.to_json()
record_sets = metadata_json.get('recordSet', [])
if isinstance(record_sets, dict):
    record_sets = [record_sets]

print(f"Found {len(record_sets)} record sets.\n")
all_fields_dict = {}
for i, recset in enumerate(record_sets):
    recset_id = recset.get('@id', f'Unknown_recordset_{i}')
    recset_name = recset.get('name', '')
    print(f"RecordSet {i+1}: @id = {recset_id}\n    Name: {recset_name}")
    # List the fields for this RecordSet
    fields = recset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    all_fields_dict[recset_id] = fields
    for field in fields:
        field_id = field.get('@id', 'unknown')
        field_name = field.get('name', '')
        print(f"      Field: @id = {field_id}, Name = {field_name}")
    print()
if len(record_sets) == 0:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from above.

For this example, we load *all* record sets found. Please modify the `record_set_ids` list or fields as appropriate for your analysis.

In [ ]:
# Extract data from all available record sets
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Get all @id's for record sets
record_set_ids = [recset.get('@id') for recset in record_sets if '@id' in recset]
dataframes = {}

for recset_id in record_set_ids:
    try:
        print(f"Loading records from RecordSet @id={recset_id} ...")
        records = list(dataset.records(record_set=recset_id))  # Each record is a dict
        if records:
            df = pd.DataFrame(records)
            dataframes[recset_id] = df
            print(f"  {len(df)} records loaded.")
            print(f"  Columns: {df.columns.tolist()}\n")
        else:
            print(f"  No records available for this RecordSet.\n")
    except Exception as e:
        print(f"  Could not load records for {recset_id}: {e}\n")

# Example: show the first 5 rows of the first dataframe if available
if dataframes:
    first_recset = list(dataframes.keys())[0]
    print(f"Displaying first 5 records of record set '{first_recset}':")
    display(dataframes[first_recset].head())
else:
    print("No dataframes could be loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

In this example, we select a numeric field (by its `@id`) from the first loaded record set. The process demonstrates basic data filtering, normalization, and grouping.

In [ ]:
# EDA: Filtering, normalization, and grouping
import numpy as np

# Choose first available record set and a numeric field
if dataframes:
    # Get the first record set and its metadata
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    fields = all_fields_dict.get(record_set_id, [])

    # Identify numeric fields (by Croissant 'dataType'), pick one
    numeric_fields = []
    for field in fields:
        dtype = field.get('dataType', '').lower()
        if dtype in ['float', 'number', 'integer']:
            numeric_fields.append(field.get('@id'))
    
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        # Try to use the '@id' as column name (as per mlcroissant convention)
        colname = numeric_field_id
        if colname not in df.columns:
            # Fallback: try with the field's 'name'
            field_obj = [f for f in fields if f.get('@id') == numeric_field_id][0]
            colname = field_obj.get('name', '')
        
        # Make sure it's numeric, coerce errors (in case values are string)
        df[colname] = pd.to_numeric(df[colname], errors='coerce')
        threshold = df[colname].quantile(0.5)  # median as threshold

        filtered_df = df[df[colname] > threshold].copy()
        print(f"Filtered records with {colname} > {threshold} (threshold is median): {len(filtered_df)} records\n")
        # Normalize
        norm_col = f"{colname}_normalized"
        filtered_df[norm_col] = (filtered_df[colname] - filtered_df[colname].mean()) / filtered_df[colname].std()
        print(filtered_df[[colname, norm_col]].head())

        # Choose a group field if possible (by Croissant convention, e.g. categorical fields)
        group_field_id = None
        for field in fields:
            dtype = field.get('dataType', '').lower()
            if dtype in ['string', 'text'] and field.get('@id') != numeric_field_id and field.get('@id') in df.columns:
                group_field_id = field.get('@id')
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[colname].mean().to_frame(name=f"mean_{colname}")
            print(f"\nMean {colname} by {group_field_id} (first 5 groups):\n", grouped_df.head())
        else:
            print("No suitable categorical group field found for grouping.")
    else:
        print("No numeric fields found in DataFrame for EDA.")
else:
    print("No data available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot a histogram of the selected numeric field for the filtered records, and a boxplot grouped by a categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'filtered_df' in locals():
    # Plot histogram
    plt.figure(figsize=(6,4))
    sns.histplot(filtered_df[colname].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {colname} (Filtered)")
    plt.xlabel(colname)
    plt.ylabel("Count")
    plt.show()

    # Boxplot grouped by group_field_id if available
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[colname])
        plt.title(f"{colname} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(colname)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a Croissant-structured dataset using the `mlcroissant` library, referencing all data elements by their `@id` values. We reviewed metadata, loaded tabular data by record set, and performed basic exploratory analysis and visualization based on those IDs.

For deeper insights, consider further analyzing the detailed structure of each record set and expanding your EDA according to research goals.